# Unified AI Portfolio: NLP & Resume Screening Engines from Scratch

This notebook contains the complete, self-contained implementation of two AI/ML projects built from scratch without using high-level ML frameworks like `scikit-learn`:

1. **Customer Feedback Sentiment & Topic Insights Dashboard** (NLP sentiment classification via Naive Bayes + unsupervised clustering via K-Means).
2. **AI Resume Screening System** (Text/PDF extraction, TF-IDF vectorization, and Cosine Similarity calculation).

## Project Highlight: No Heavy External ML Libraries
All vectorizers (`TfidfVectorizer`), classifiers (`Naive Bayes`), and clustering algorithms (`K-Means`) are built using **pure Python and NumPy**. This proves a solid foundation in the mathematical mechanics behind these models!

In [ ]:
import re
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)

## Text Preprocessing & Tokenization
We define a custom clean and tokenize helper that removes punctuation, lowercases the text, and filters out standard English stop words.

In [ ]:
# Stop words list for cleaning text
STOP_WORDS = {
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 'your', 'yours', 
    'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', 'her', 'hers', 
    'herself', 'it', 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 
    'what', 'which', 'who', 'whom', 'this', 'that', 'these', 'those', 'am', 'is', 'are', 
    'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 
    'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 
    'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 
    'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 
    'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 
    'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 
    'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 
    'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', 'should', 'now', 'd', 
    'll', 'm', 'o', 're', 've', 'y', 'ain', 'aren', 'couldn', 'didn', 'doesn', 'hadn', 
    'hasn', 'haven', 'isn', 'ma', 'mightn', 'mustn', 'needn', 'shan', 'shouldn', 'wasn', 
    'weren', 'won', 'wouldn'
}

def clean_and_tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    return [t for t in tokens if t not in STOP_WORDS and len(t) > 1]

## 1. Custom TF-IDF Vectorizer
This class implements the classic **Term Frequency - Inverse Document Frequency** formulation:
- TF (Term Frequency) with log normalization: `1 + log(tf)` if `tf > 0` else `0`.
- IDF (Inverse Document Frequency) with smooth formulation: `log((1 + n_samples) / (1 + df)) + 1`.
- L2 normalization is applied to each sample vector.

In [ ]:
class PureTfidfVectorizer:
    def __init__(self, max_features=1000):
        self.max_features = max_features
        self.vocabulary_ = {}
        self.feature_names_ = []
        self.idf_ = []
        
    def fit(self, raw_documents):
        df = {}
        n_samples = len(raw_documents)
        for doc in raw_documents:
            tokens = set(clean_and_tokenize(doc))
            for token in tokens:
                df[token] = df.get(token, 0) + 1
        sorted_words = sorted(df.items(), key=lambda x: x[1], reverse=True)
        top_words = sorted_words[:self.max_features]
        self.vocabulary_ = {word[0]: idx for idx, word in enumerate(top_words)}
        self.feature_names_ = [word[0] for word in top_words]
        self.idf_ = []
        for word in self.feature_names_:
            word_df = df[word]
            idf = np.log((1 + n_samples) / (1 + word_df)) + 1.0
            self.idf_.append(idf)
        self.idf_ = np.array(self.idf_)
        return self
        
    def transform(self, raw_documents):
        n_samples = len(raw_documents)
        n_features = len(self.feature_names_)
        X = np.zeros((n_samples, n_features))
        for i, doc in enumerate(raw_documents):
            tokens = clean_and_tokenize(doc)
            if not tokens:
                continue
            tf = {}
            for t in tokens:
                if t in self.vocabulary_:
                    tf[t] = tf.get(t, 0) + 1
            for word, freq in tf.items():
                idx = self.vocabulary_[word]
                tf_val = 1 + np.log(freq) if freq > 0 else 0
                X[i, idx] = tf_val * self.idf_[idx]
            norm = np.linalg.norm(X[i])
            if norm > 0:
                X[i] = X[i] / norm
        return X

    def fit_transform(self, raw_documents):
        return self.fit(raw_documents).transform(raw_documents)
        
    def get_feature_names_out(self):
        return np.array(self.feature_names_)

## 2. Custom Naive Bayes Sentiment Classifier
This class implements a multinomial Naive Bayes classifier:
- Prior probabilities are computed in log-space: `log_prior = log(class_count / total_samples)`.
- Conditional likelihoods are computed with Laplace smoothing: `log_prob = log((count + alpha) / (total_class_count + alpha * n_features))`.
- Predictions use dot product summation and softmax normalization to yield stable probability scores.

In [ ]:
class PureNaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes = None
        self.class_log_prior_ = None
        self.feature_log_prob_ = None
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        class_count = np.zeros(n_classes)
        feature_count = np.zeros((n_classes, n_features))
        for c_idx, c in enumerate(self.classes):
            mask = (y == c)
            class_count[c_idx] = np.sum(mask)
            feature_count[c_idx] = np.sum(X[mask], axis=0)
        self.class_log_prior_ = np.log(class_count / n_samples)
        smoothed_feature_count = feature_count + self.alpha
        smoothed_class_feature_sum = np.sum(smoothed_feature_count, axis=1, keepdims=True)
        self.feature_log_prob_ = np.log(smoothed_feature_count / smoothed_class_feature_sum)
        return self
        
    def predict_proba(self, X):
        jlog = X @ self.feature_log_prob_.T + self.class_log_prior_
        jlog_max = np.max(jlog, axis=1, keepdims=True)
        exp_jlog = np.exp(jlog - jlog_max)
        probs = exp_jlog / np.sum(exp_jlog, axis=1, keepdims=True)
        return probs
        
    def predict(self, X):
        probs = self.predict_proba(X)
        indices = np.argmax(probs, axis=1)
        return self.classes[indices]

## 3. Custom K-Means Topic Clustering
This class implements the classic K-Means unsupervised clustering algorithm:
- Initializes cluster centers by randomly selecting data points.
- Assigns labels by calculating Euclidean distances to all centroids.
- Recalculates centroids by computing the mean of all points assigned to each cluster.
- Iterates until convergence (where centroid movements drop below a tiny tolerance threshold).

In [ ]:
class PureKMeans:
    def __init__(self, n_clusters=4, max_iter=100, random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.cluster_centers_ = None
        
    def fit(self, X):
        n_samples, n_features = X.shape
        rng = np.random.RandomState(self.random_state)
        indices = rng.permutation(n_samples)[:self.n_clusters]
        self.cluster_centers_ = X[indices].copy()
        for _ in range(self.max_iter):
            distances = np.zeros((n_samples, self.n_clusters))
            for c_idx in range(self.n_clusters):
                distances[:, c_idx] = np.linalg.norm(X - self.cluster_centers_[c_idx], axis=1)
            labels = np.argmin(distances, axis=1)
            new_centers = np.zeros_like(self.cluster_centers_)
            for c_idx in range(self.n_clusters):
                mask = (labels == c_idx)
                if np.sum(mask) > 0:
                    new_centers[c_idx] = np.mean(X[mask], axis=0)
                else:
                    new_centers[c_idx] = self.cluster_centers_[c_idx]
            if np.allclose(self.cluster_centers_, new_centers, atol=1e-6):
                break
            self.cluster_centers_ = new_centers
        return self
        
    def predict(self, X):
        n_samples = X.shape[0]
        distances = np.zeros((n_samples, self.n_clusters))
        for c_idx in range(self.n_clusters):
            distances[:, c_idx] = np.linalg.norm(X - self.cluster_centers_[c_idx], axis=1)
        return np.argmin(distances, axis=1)

## 4. Dataset Generation (E-Commerce Reviews)
Let's generate 1000 reviews across different categories (Electronics, Clothing, Kitchen, Books) and sentiments (positive, neutral, negative), injecting key terms representing shipping, quality, price, and customer service.

In [ ]:
# Setup data generator
categories = ["Electronics", "Clothing & Fashion", "Home & Kitchen", "Books & Media"]

review_templates = {
    "positive": [
        "Absolutely love this product! The quality is outstanding and exceeded my expectations.",
        "Best purchase I have made in a long time. Highly recommend to everyone.",
        "Excellent product. It works perfectly and the design is very sleek and modern.",
        "Very satisfied with this purchase. High quality material and works like a charm.",
        "Great value for money. It does exactly what it says and feels premium.",
        "Super fast shipping! The item arrived in perfect condition and is top notch.",
        "Five stars! The customer service was also very helpful and responsive.",
        "Very easy to set up and use. The build quality feels sturdy and durable.",
        "I was skeptical at first, but this is brilliant. Definitely worth the price.",
        "Perfect fit and works beautifully. Will definitely buy from this brand again."
    ],
    "neutral": [
        "It is decent for the price. Not amazing, but does the job fine.",
        "Average product. It works as described, but the build quality could be better.",
        "It is okay. Shipping was a bit slow, but the product is acceptable.",
        "Neutral opinion. It has some good features but also a few design flaws.",
        "The product works, but the customer service was not very helpful.",
        "Okay product, but I feel like it is slightly overpriced for what it offers.",
        "Not bad, not great. Standard product that meets basic expectations.",
        "It performs okay, but there are better alternatives on the market.",
        "The size is smaller than expected, but the quality is fine otherwise.",
        "Delivery was quick, but the product packaging was a bit damaged."
    ],
    "negative": [
        "Terrible quality. It broke within the first day of use. Do not buy!",
        "Very disappointed. The product looks cheap and does not work properly.",
        "Waste of money. Extremely slow shipping and the item is defective.",
        "Worst experience ever. The product didn't match the description at all.",
        "Poor build quality. The plastic feels cheap and it makes a weird noise.",
        "The customer service was awful, and the product arrived damaged.",
        "I would not recommend this. It stopped working after a week of light use.",
        "Very overpriced for such poor quality. I am returning it immediately.",
        "Frustrating to use. The instructions are unclear and it is very fragile.",
        "Horrible purchase. Avoid this seller and product. Save your money."
    ]
}

topic_keywords = {
    "shipping": ["shipping", "delivery", "arrived", "packaging", "shipped", "carrier", "late", "fast"],
    "quality": ["quality", "build", "material", "durable", "sturdy", "cheap", "plastic", "broke", "defective"],
    "price": ["price", "value", "money", "overpriced", "cost", "cheap", "expensive", "deal", "worth"],
    "service": ["service", "customer", "support", "return", "refund", "seller", "contact", "helpful"]
}

def generate_reviews(num_records=1000):
    data = []
    base_date = datetime(2026, 1, 1)
    for i in range(num_records):
        sentiment = random.choices(["positive", "neutral", "negative"], weights=[0.5, 0.2, 0.3])[0]
        base_text = random.choice(review_templates[sentiment])
        topic_target = random.choice(["shipping", "quality", "price", "service"])
        keyword = random.choice(topic_keywords[topic_target])
        
        filler_phrases = [
            f" Regarding the {keyword}, I must say it was notable.",
            f" The {keyword} aspect could be improved.",
            f" I was particularly focused on the {keyword}.",
            f" Standard {keyword} experience overall.",
            f" Especially satisfied with the {keyword}."
        ]
        review_text = base_text + random.choice(filler_phrases)
        
        if sentiment == "positive":
            rating = random.choices([5, 4], weights=[0.8, 0.2])[0]
        elif sentiment == "neutral":
            rating = random.choices([3, 4, 2], weights=[0.7, 0.15, 0.15])[0]
        else:
            rating = random.choices([1, 2], weights=[0.7, 0.3])[0]
            
        category = random.choice(categories)
        date = base_date + timedelta(days=random.randint(0, 150))
        helpful_votes = random.randint(0, 50)
        
        data.append({
            "review_id": f"REV_{i+1:04d}",
            "category": category,
            "rating": rating,
            "review_text": review_text,
            "sentiment": sentiment,
            "helpful_votes": helpful_votes,
            "date": date.strftime("%Y-%m-%d")
        })
    return pd.DataFrame(data)

df = generate_reviews(1000)
df["clean_text"] = df["review_text"].apply(lambda x: " ".join(clean_and_tokenize(x)))
print(f"Generated {len(df)} sample reviews. Ready for NLP training!")
df.head()

## 5. Training Sentiment Classifier & Topic Clustering Models
Let's run a train-test split, transform text using `PureTfidfVectorizer`, and train the `PureNaiveBayes` model and evaluate its performance. After that, we'll run K-Means on the dataset to cluster reviews into 4 core feedback channels.

In [ ]:
# 1. Train-test split
def pure_train_test_split(X, y, test_size=0.2, random_state=42):
    np.random.seed(random_state)
    shuffled_indices = np.random.permutation(len(X))
    split_idx = int(len(X) * (1.0 - test_size))
    train_idx = shuffled_indices[:split_idx]
    test_idx = shuffled_indices[split_idx:]
    return X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]

X_train, X_test, y_train, y_test = pure_train_test_split(df["clean_text"], df["sentiment"], test_size=0.2, random_state=42)

# 2. Vectorize
vec = PureTfidfVectorizer(max_features=1000)
X_train_vec = vec.fit_transform(X_train.tolist())
X_test_vec = vec.transform(X_test.tolist())

# 3. Train Naive Bayes
clf = PureNaiveBayes()
clf.fit(X_train_vec, y_train.to_numpy())

# 4. Evaluate
y_pred = clf.predict(X_test_vec)
accuracy = np.mean(y_pred == y_test.to_numpy())
print(f"--- Sentiment Classifier Results ---")
print(f"Overall Accuracy: {accuracy:.4f}\n")
for c in np.unique(y_test):
    tp = np.sum((y_test == c) & (y_pred == c))
    fp = np.sum((y_test != c) & (y_pred == c))
    fn = np.sum((y_test == c) & (y_pred != c))
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    print(f"Class: {c:10s} | Precision: {p:.4f} | Recall: {r:.4f} | F1: {f1:.4f}")

# 5. Topic Clustering with K-Means
print("\n--- Training Unsupervised K-Means Topic Clustering ---")
X_all_vec = vec.transform(df["clean_text"].tolist())
kmeans = PureKMeans(n_clusters=4, random_state=42)
kmeans.fit(X_all_vec)

terms = vec.get_feature_names_out()
for i in range(4):
    centroid = kmeans.cluster_centers_[i]
    order_centroids = centroid.argsort()[::-1]
    top_words = [terms[ind] for ind in order_centroids[:8]]
    print(f"Cluster {i} top keywords: {', '.join(top_words)}")

## 6. AI Resume Screening Engine
The AI Resume Screening engine matches candidate resumes against a Job Description. 
- It uses our custom TF-IDF Vectorizer to vectorise the union of the job description and candidate resume texts.
- It computes the **Cosine Similarity** between the Job Description vector and each candidate's vector.
- It extracts matching and missing technical skills using **regex boundary assertions** that safely match special tech symbols (e.g. `C++`, `C#`).

In [ ]:
# Common technical skills list
SKILLS_LIST = [
    "python", "javascript", "typescript", "java", "c++", "c#", "go", "rust", "ruby", "php", "sql", "html", "css",
    "machine learning", "deep learning", "nlp", "computer vision", "tensorflow", "pytorch", "keras", 
    "scikit-learn", "numpy", "pandas", "scipy", "transformers", "huggingface", "llm", "rag", "embeddings",
    "react", "angular", "vue", "next.js", "vite", "nodejs", "express", "fastapi", "flask", "django", 
    "docker", "kubernetes", "aws", "azure", "gcp", "firebase", "mongodb", "postgresql", "mysql", "redis",
    "git", "github", "agile", "jira", "scrum", "jenkins", "ci/cd"
]

def screen_candidates(job_description, candidates):
    """
    Screens resumes against a Job Description.
    candidates: list of dicts with {"name": str, "text": str}
    """
    resume_texts = [c["text"] for c in candidates]
    all_docs = [job_description] + resume_texts
    
    # Run TF-IDF Vectorizer
    vectorizer_temp = PureTfidfVectorizer(max_features=1000)
    vectors = vectorizer_temp.fit_transform(all_docs)
    
    jd_vector = vectors[0]
    resume_vectors = vectors[1:]
    
    results = []
    for idx, cand in enumerate(candidates):
        res_vector = resume_vectors[idx]
        
        # L2 normalized dot product is the cosine similarity score
        similarity_score = float(np.dot(jd_vector, res_vector))
        
        jd_lower = job_description.lower()
        res_lower = cand["text"].lower()
        
        matching_skills = []
        missing_skills = []
        
        for skill in SKILLS_LIST:
            # Custom boundary checks for symbols C++ and C#
            if skill == "c++":
                pattern = r'\bc\+\+(?=[^a-zA-Z0-9]|$)'
            elif skill == "c#":
                pattern = r'\bc#(?=[^a-zA-Z0-9]|$)'
            else:
                pattern = r'\b' + re.escape(skill) + r'\b'
                
            if re.search(pattern, jd_lower):
                if re.search(pattern, res_lower):
                    matching_skills.append(skill)
                else:
                    missing_skills.append(skill)
                    
        results.append({
            "name": cand["name"],
            "score": similarity_score,
            "matching_skills": matching_skills,
            "missing_skills": missing_skills
        })
        
    # Sort candidates by score descending
    results = sorted(results, key=lambda x: x["score"], reverse=True)
    return results

## 7. Resume Screener Demo
Let's define a Job Description for an **AI/Machine Learning Engineer** and three sample candidate resumes:
- **Alice Smith**: Strong Python, PyTorch, Scikit-Learn, Pandas, Git, and Machine Learning.
- **Bob Johnson**: Java developer with SQL, Docker, AWS, and Git.
- **Charlie Brown**: Web developer with React, JavaScript, Node.js, and CSS.

In [ ]:
job_description = """
We are seeking a talented AI/Machine Learning Engineer to join our team.
Requirements:
- Strong programming experience in Python.
- Completed coursework or projects involving Machine Learning and NLP.
- Hands-on experience with Scikit-Learn, NumPy, and Pandas.
- Familiarity with deep learning frameworks like PyTorch or TensorFlow.
- Experience with FastAPI, Git, Github, and Agile methodologies.
"""

candidates = [
    {
        "name": "Alice Smith (ML Engineer Resume)",
        "text": """
        Alice Smith
        EXPERIENCE:
        Machine Learning Engineer Intern - Tech Corp
        - Built NLP models using Python, NumPy, and Pandas.
        - Deployed neural networks with PyTorch and Scikit-Learn for text classification.
        - Managed repositories with Git and GitHub, working in an Agile scrum team.
        SKILLS: Python, PyTorch, Scikit-Learn, NumPy, Pandas, Git, GitHub, Agile, Machine Learning, NLP
        """
    },
    {
        "name": "Bob Johnson (Backend Java Resume)",
        "text": """
        Bob Johnson
        EXPERIENCE:
        Software Engineer - Systems LLC
        - Maintained enterprise applications written in Java.
        - Wrote SQL queries and structured relational databases using PostgreSQL.
        - Packaged microservices inside Docker containers and deployed to AWS.
        SKILLS: Java, SQL, Docker, Kubernetes, AWS, Git, Jira, Agile
        """
    },
    {
        "name": "Charlie Brown (Frontend Web Resume)",
        "text": """
        Charlie Brown
        EXPERIENCE:
        Frontend Web Developer - Creative Agency
        - Designed user interfaces with React, JavaScript, HTML, and CSS.
        - Optimized build pipelines using Vite and npm packages.
        - Collaborated on styling using TailwindCSS and styled-components.
        SKILLS: React, JavaScript, TypeScript, HTML, CSS, Vite, Node.js, Git
        """
    }
]

# Run screening
rankings = screen_candidates(job_description, candidates)

print("--- Candidate Screener Match Rankings ---")
for r in rankings:
    print(f"Candidate: {r['name']}")
    print(f"Cosine Similarity Match Score: {r['score']:.4f}")
    print(f"Matching Skills found: {r['matching_skills']}")
    print(f"Missing Skills: {r['missing_skills']}")
    print("-" * 50)
